In [0]:
#2. Spark Coding Questions Given two tables: 
# Employee and Salary Write Spark code to: 
# Join the Employee and Salary tables. 
# Find employees whose salary is greater than ₹100,000. 
# Find null values in the Employee table. Extract the first 3 characters (or words) from the employee name. 
# Explain the logic behind each solution and why you used that approach.

In [0]:
from pyspark.sql.functions import *

employee_data = [
    (101, "Vinay Kumar", "IT", "TCS"),
    (102, "Rahul Sharma", "HR", "Infosys"),
    (103, "Anu Priya", "Finance", "Wipro"),
    (104, None, "IT", "TCS"),
    (105, "Kiran Kumar", None, "Accenture")
]

employee_cols = [
    "emp_id",
    "emp_name",
    "department",
    "company_name"
]

employee_df = spark.createDataFrame(
    employee_data,
    employee_cols
)

employee_df.show(truncate=False)



In [0]:
del employee_df

In [0]:
salary_data = [
    (101, 120000),
    (102, 85000),
    (103, 150000),
    (104, 95000),
    (105, 110000)
]

salary_cols = ["emp_id", "salary"]

salary_df = spark.createDataFrame(salary_data, salary_cols)

salary_df.show()

In [0]:
# JOIN
Join_df = employee_df.join(salary_df, employee_df.emp_id == salary_df.emp_id, "left")
display(Join_df)

In [0]:
# Find employees whose salary is greater than ₹100,000. 
# Find null values in the Employee table. 
# Extract the first 3 characters (or words) from the employee name. 
# Explain the logic behind each solution and why you used that approach.
filter_df = Join_df.filter(col("salary") > 100000) \
                   .filter(col("department").isNotNull()) \
                   .withColumn("first_3_chars",substring(col("company_name"), 1, 3)
                   )
display(filter_df)

In [0]:
%sql
CREATE TABLE employees_new(
    emp_id INT,
    emp_name VARCHAR(100),
    department VARCHAR(100),
    salary INT,
    join_date DATE
);

INSERT INTO employees_new VALUES 
(1, 'Aarav','IT',120000,'2019-06-10'),
(2, 'Meera','HR',90000,'2020-03-15'),
(3, 'Rahul','IT',110000,'2020-11-20'),
(4, 'Sneha','Finance',95000,'2021-01-05'),
(5, 'Aditi','HR',98000,'2020-07-01'),
(6, 'Vikram','Sales',85000,'2018-09-12');

In [0]:
employee_data = [
    (1, "Aarav", "IT", 120000, "2019-06-10"),
    (2, "Meera", "HR", 90000, "2020-03-15"),
    (3, "Rahul", "IT", 110000, "2020-11-20"),
    (4, "Sneha", "Finance", 95000, "2021-01-05"),
    (5, "Aditi", "HR", 98000, "2020-07-01"),
    (6, "Vikram", "Sales", 85000, "2018-09-12")
]

cols = ["emp_id","emp_name","department","salary","join_datee"]

df = spark.createDataFrame(employee_data,cols)
display(df)

In [0]:
#HR or IT
from pyspark.sql.functions import *
dep_df = df.filter(col("department").isin("HR","IT")).show()

# total salay
total_sal_df = df.agg(sum(col("salary")).alias("total_sal")).show()


# Note : in spark opposite to "isin" is "~"
df.filter(~col("department").isin("IT", "HR")).show()

In [0]:
%sql
SELECT * FROM employees_new;
-- ## get the employees who are in the HR OR IT department
SELECT emp_id, emp_name,department
FROM employees_new
-- WHERE department = 'IT' OR department = 'HR'; (normal way)
WHERE department IN ('HR','IT')

In [0]:
%sql
-- Q) Total salary paid
SELECT SUM(salary) AS Total_salary
FROM employees_new

In [0]:
%sql
-- Q) Employee joined in 2020
SELECT *
FROM employees_new
WHERE join_date BETWEEN '2020-01-01' AND '2020-12-31';

SELECT * FROM employees_new
WHERE join_date >= '2020-01-01' AND join_date <= '2020-12-31';

SELECT * FROM employees_new
WHERE YEAR(join_date) = '2020';
-- (we can use join_date <= or we can use one day extra < 2021-01-01) 

In [0]:
%sql
-- Q) Name starts with A
SELECT * FROM employees_new
WHERE emp_name LIKE 'A%'

In [0]:
from pyspark.sql.functions import *
data = [
    (1, [101, 102, 103]),
    (2, [201, 202])
]

df = spark.createDataFrame(data, ["customer_id", "orders"])

df.show(truncate=False)

In [0]:
#I want the orders column should come properly
final_df = df.select(col("customer_id"), explode("orders").alias("orders")).show()

In [0]:
# example for window fun
data = [
    (101, "Vinay", 50000),
    (102, "Rahul", 70000),
    (103, "Anu", 60000),
    (104, "Kiran", 70000)
]

df = spark.createDataFrame(data, ["emp_id", "emp_name", "salary"])

In [0]:
# ans
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

window_spec = Window.orderBy(col("salary").desc())

final_df = df.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

final_df.show()

new_df = final_df.withColumnRenamed("rank","new_rank").show()